In [5]:
import random
import math
import time

# ==========================================
# DEFINING THE CUBE AND VECTOR ROTATIONS
# ==========================================
# 3D coordinates are (x, y, z) in {-1, 1}
# x: -1=Left, 1=Right
# y: -1=Back, 1=Front
# z: -1=Down, 1=Up
# Note: DBL corner is (-1, -1, -1).

stickers = [
    ((-1,-1, 1), ( 0, 0, 1)), ((-1,-1, 1), ( 0,-1, 0)), ((-1,-1, 1), (-1, 0, 0)), # UBL
    (( 1,-1, 1), ( 0, 0, 1)), (( 1,-1, 1), ( 1, 0, 0)), (( 1,-1, 1), ( 0,-1, 0)), # UBR
    (( 1, 1, 1), ( 0, 0, 1)), (( 1, 1, 1), ( 0, 1, 0)), (( 1, 1, 1), ( 1, 0, 0)), # UFR
    ((-1, 1, 1), ( 0, 0, 1)), ((-1, 1, 1), (-1, 0, 0)), ((-1, 1, 1), ( 0, 1, 0)), # UFL
    ((-1, 1,-1), ( 0, 0,-1)), ((-1, 1,-1), ( 0, 1, 0)), ((-1, 1,-1), (-1, 0, 0)), # DFL
    (( 1, 1,-1), ( 0, 0,-1)), (( 1, 1,-1), ( 1, 0, 0)), (( 1, 1,-1), ( 0, 1, 0)), # DFR
    (( 1,-1,-1), ( 0, 0,-1)), (( 1,-1,-1), ( 0,-1, 0)), (( 1,-1,-1), ( 1, 0, 0))  # DBR
]

def rot_U(pos, norm):
    if pos[2] == 1: return (-pos[1], pos[0], pos[2]), (-norm[1], norm[0], norm[2])
    return pos, norm

def rot_R(pos, norm):
    if pos[0] == 1: return (pos[0], -pos[2], pos[1]), (norm[0], -norm[2], norm[1])
    return pos, norm

def rot_F(pos, norm):
    if pos[1] == 1: return (pos[2], pos[1], -pos[0]), (norm[2], norm[1], -norm[0])
    return pos, norm

def get_permutation(rot_func):
    perm = []
    for st in stickers:
        new_st = rot_func(st[0], st[1])
        perm.append(stickers.index(new_st))
    return tuple(perm)

def apply_move(p1, p2):
    return tuple(p2[p1[i]] for i in range(21))

# Generate base moves
U_move = get_permutation(rot_U)
R_move = get_permutation(rot_R)
F_move = get_permutation(rot_F)

# Generate double moves
U2 = apply_move(U_move, U_move)
R2 = apply_move(R_move, R_move)
F2 = apply_move(F_move, F_move)

# Inverse move optimization is skipped
moves_to_opt = [U_move, U2, R_move, R2, F_move, F2]
move_names = ["U", "U2", "R", "R2", "F", "F2"]


# ==========================================
# SIMULATED ANNEALING OPTIMIZATION
# ==========================================
def cost(L):
    max_c, sum_c = 0, 0
    for p in moves_to_opt:
        shifts = set()
        for src in range(21):
            dst = p[src]
            if src != dst:
                shifts.add(L[dst] - L[src])
        c = len(shifts)
        if c > max_c: max_c = c
        sum_c += c
    return max_c, sum_c

print("Starting layout optimization...")
start_time = time.time()

best_global_L = None
best_global_cost = (999, 999)

RESTARTS = 200
STEPS_PER_RESTART = 100000

for run in range(RESTARTS):
    current_L = list(range(21))
    random.shuffle(current_L)

    current_cost = cost(current_L)
    best_local_L = current_L[:]
    best_local_cost = current_cost

    T = 10.0
    alpha = 0.9999

    for step in range(STEPS_PER_RESTART):
        i, j = random.sample(range(21), 2)
        current_L[i], current_L[j] = current_L[j], current_L[i]

        new_cost = cost(current_L)

        if new_cost < current_cost:
            current_cost = new_cost
            if new_cost < best_local_cost:
                best_local_cost = new_cost
                best_local_L = current_L[:]
        else:
            diff = (new_cost[0]*100 + new_cost[1]) - (current_cost[0]*100 + current_cost[1])
            if math.exp(-diff / T) > random.random():
                current_cost = new_cost
            else:
                current_L[i], current_L[j] = current_L[j], current_L[i]
        T *= alpha

    if best_local_cost < best_global_cost:
        best_global_cost = best_local_cost
        best_global_L = best_local_L[:]

    print(f"Restart {run+1}/{RESTARTS} best: Max Shifts={best_local_cost[0]}, Sum Shifts={best_local_cost[1]}")

print(f"\nOptimization Finished in {time.time() - start_time:.2f} seconds")
print(f"Found layout with Worst-Case Move Cost = {best_global_cost[0]} shifts")


# ==========================================
# 3. CODE GENERATOR
# ==========================================
def generate_cpp_move(name, perm, L):
    shifts = {}
    stationary = []
    for src in range(21):
        dst = perm[src]
        if src == dst:
            stationary.append(L[src])
        else:
            shift = L[dst] - L[src]
            if shift not in shifts:
                shifts[shift] = []
            shifts[shift].append(L[src])

    lines = [f"// --- Move {name} ---"]
    lines.append(f"uint64_t move_{name}(uint64_t state) {{")

    stat_mask = sum((7 << (3 * slot)) for slot in stationary)
    if stat_mask != 0:
        lines.append(f"    uint64_t new_state = state & 0x{stat_mask:016X}ULL;")
    else:
        lines.append(f"    uint64_t new_state = 0;")

    for shift, slots in sorted(shifts.items()):
        mask = sum((7 << (3 * slot)) for slot in slots)
        if shift > 0:
            lines.append(f"    new_state |= (state & 0x{mask:016X}ULL) << {shift * 3};")
        else:
            lines.append(f"    new_state |= (state & 0x{mask:016X}ULL) >> {-shift * 3};")

    lines.append("    return new_state;")
    lines.append("}")
    return "\n".join(lines)

print("\n" + "="*42)
print("BEST MEMORY LAYOUT FOUND:")
print("="*42)
print("Physical Sticker ID -> 3-bit Memory Slot Index")
for i, slot in enumerate(best_global_L):
    print(f"Sticker {i:02d} -> Slot {slot:02d}")

Starting layout optimization...
Restart 1/200 best: Max Shifts=6, Sum Shifts=29
Restart 2/200 best: Max Shifts=5, Sum Shifts=24
Restart 3/200 best: Max Shifts=6, Sum Shifts=29
Restart 4/200 best: Max Shifts=6, Sum Shifts=27
Restart 5/200 best: Max Shifts=5, Sum Shifts=24
Restart 6/200 best: Max Shifts=7, Sum Shifts=35
Restart 7/200 best: Max Shifts=6, Sum Shifts=26
Restart 8/200 best: Max Shifts=6, Sum Shifts=27
Restart 9/200 best: Max Shifts=6, Sum Shifts=29
Restart 10/200 best: Max Shifts=6, Sum Shifts=35
Restart 11/200 best: Max Shifts=6, Sum Shifts=34
Restart 12/200 best: Max Shifts=7, Sum Shifts=39
Restart 13/200 best: Max Shifts=6, Sum Shifts=33
Restart 14/200 best: Max Shifts=7, Sum Shifts=38
Restart 15/200 best: Max Shifts=6, Sum Shifts=28
Restart 16/200 best: Max Shifts=7, Sum Shifts=31
Restart 17/200 best: Max Shifts=5, Sum Shifts=26
Restart 18/200 best: Max Shifts=7, Sum Shifts=37
Restart 19/200 best: Max Shifts=6, Sum Shifts=31
Restart 20/200 best: Max Shifts=6, Sum Shifts=

In [6]:
# ==========================================
# FOUND OPTIMIZED LAYOUT
# ==========================================
# Note that this layout is different and was only updated at the run that went into the final C++ code
best_layout = [1, 2, 0, 10, 11, 12, 9, 7, 8, 5, 6, 4, 20, 18, 19, 16, 17, 15, 3, 13, 14]

# ==========================================
# ASCII MANIFOLD GENERATOR
# ==========================================
def print_manifold(L):
    U_face = [0, 3, 9, 6]       # UBL, UBR, UFL, UFR
    F_face = [11, 7, 13, 17]    # UFL, UFR, DFL, DFR
    R_face = [8, 4, 16, 20]     # UFR, UBR, DFR, DBR
    B_face = [5, 1, 19, -1]     # UBR, UBL, DBR, DBL
    L_face = [2, 10, -1, 14]    # UBL, UFL, DBL, DFL
    D_face = [12, 15, -1, 18]   # DFL, DFR, DBL, DBR

    def fmt(idx):
        return f"{L[idx]:02d}" if idx != -1 else "xx"

    print("="*42)
    print("ASCII MANIFOLD: STICKER TO MEMORY SLOTS")
    print("="*42)
    print("             +-------+")
    print(f"             | {fmt(U_face[0])} {fmt(U_face[1])} |")
    print(f"             | {fmt(U_face[2])} {fmt(U_face[3])} |")
    print("     +-------+-------+-------+-------+")
    print(f"     | {fmt(L_face[0])} {fmt(L_face[1])} | {fmt(F_face[0])} {fmt(F_face[1])} | {fmt(R_face[0])} {fmt(R_face[1])} | {fmt(B_face[0])} {fmt(B_face[1])} |")
    print(f"     | {fmt(L_face[2])} {fmt(L_face[3])} | {fmt(F_face[2])} {fmt(F_face[3])} | {fmt(R_face[2])} {fmt(R_face[3])} | {fmt(B_face[2])} {fmt(B_face[3])} |")
    print("     +-------+-------+-------+-------+")
    print(f"             | {fmt(D_face[0])} {fmt(D_face[1])} |")
    print(f"             | {fmt(D_face[2])} {fmt(D_face[3])} |")
    print("             +-------+")
    print("        (xx = fixed DBL corner)")


# ==========================================
# 3. CODE GENERATOR
# ==========================================
def generate_cpp_move(name, perm, L):
    shifts = {}
    stationary = []

    for src in range(21):
        dst = perm[src]
        if src == dst:
            stationary.append(L[src])
        else:
            shift = L[dst] - L[src]
            if shift not in shifts:
                shifts[shift] = []
            shifts[shift].append(L[src])

    func_name = name.replace("'", "_prime")
    lines = [f"// --- {name} Sequence ---"]
    lines.append(f"uint64_t move_{func_name}(uint64_t state) {{")

    stat_mask = sum((7 << (3 * slot)) for slot in stationary)
    if stat_mask != 0:
        lines.append(f"    uint64_t new_state = state & 0x{stat_mask:016X}ULL;")
    else:
        lines.append(f"    uint64_t new_state = 0;")

    for shift, slots in sorted(shifts.items()):
        mask = sum((7 << (3 * slot)) for slot in slots)
        if shift > 0:
            lines.append(f"    new_state |= (state & 0x{mask:016X}ULL) << {shift * 3};")
        else:
            lines.append(f"    new_state |= (state & 0x{mask:016X}ULL) >> {-shift * 3};")

    lines.append("    return new_state;")
    lines.append("}\n")
    return "\n".join(lines)


# Execute and Print Output
print_manifold(best_layout)

print("\n" + "="*42)
print("GENERATED C++ BITHACK CODE:")
print("="*42)

for move_name, move_perm in all_9_moves:
    print(generate_cpp_move(move_name, move_perm, best_layout))

ASCII MANIFOLD: STICKER TO MEMORY SLOTS
             +-------+
             | 01 10 |
             | 05 09 |
     +-------+-------+-------+-------+
     | 00 06 | 04 07 | 08 11 | 12 02 |
     | xx 19 | 18 15 | 17 14 | 13 xx |
     +-------+-------+-------+-------+
             | 20 16 |
             | xx 03 |
             +-------+
        (xx = fixed DBL corner)

GENERATED C++ BITHACK CODE:
// --- U Sequence ---
uint64_t move_U(uint64_t state) {
    uint64_t new_state = state & 0x7FFFFF8000000E00ULL;
    new_state |= (state & 0x0000007E3F1FF000ULL) >> 12;
    new_state |= (state & 0x00000001C0E00000ULL) >> 3;
    new_state |= (state & 0x00000000000001F8ULL) << 27;
    new_state |= (state & 0x0000000000000007ULL) << 36;
    return new_state;
}

// --- U' Sequence ---
uint64_t move_U_prime(uint64_t state) {
    uint64_t new_state = state & 0x7FFFFF8000000E00ULL;
    new_state |= (state & 0x0000007000000000ULL) >> 36;
    new_state |= (state & 0x0000000FC0000000ULL) >> 27;
    new_state 